## C 语言版说明

本 notebook 对应 **C 语言提交版本**。你不需要在这里写任何 Python 代码。

请将你的实现写在 `solution.c` 中，函数签名见 `integration.h`。  
提交时将本 notebook（不需要修改）和 `solution.c` 一起放在提交目录下即可。

---

## 下面这个 cell 勿删，会自动编译 solution.c 并加载

In [ ]:
import ctypes
import pathlib
import subprocess
from typing import Callable

import mpmath as mpm
import numpy as np
import sympy as sp

mpm.mp.dps = 50

# ── 编译 solution.c → solution.so，并暴露与 Python 版同名的包装函数 ──────────
_CFUNC = ctypes.CFUNCTYPE(ctypes.c_double, ctypes.c_double)


def _load_solution(c_file="solution.c"):
    src = pathlib.Path(c_file)
    so = src.with_suffix(".so")
    result = subprocess.run(
        ["gcc", "-shared", "-fPIC", "-O2", "-o", str(so), str(src), "-lm"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"编译失败:\n{result.stderr}")
    lib = ctypes.CDLL(str(so.resolve()))
    for _name in ("composite_trapezoid", "composite_simpson"):
        fn = getattr(lib, _name)
        fn.argtypes = [_CFUNC, ctypes.c_double, ctypes.c_double, ctypes.c_int]
        fn.restype = ctypes.c_double
    lib.romberg.argtypes = [_CFUNC, ctypes.c_double, ctypes.c_double, ctypes.c_double]
    lib.romberg.restype = ctypes.c_double
    lib.gauss_legendre.argtypes = [_CFUNC, ctypes.c_double, ctypes.c_double, ctypes.c_int]
    lib.gauss_legendre.restype = ctypes.c_double
    return lib


_lib = _load_solution()


# 包装函数：签名与 Python 版完全一致，grading cell 无需任何修改
# 注意：_f = _CFUNC(f) 须在调用期间保持引用，防止 GC 提前回收函数指针
def composite_trapezoid(f: Callable[[float], float], a: float, b: float, m: int = 8) -> float:
    """使用 m 个分点的复化梯形公式计算定积分（调用 C 实现）"""
    _f = _CFUNC(f)
    return _lib.composite_trapezoid(_f, a, b, m)


def composite_simpson(f: Callable[[float], float], a: float, b: float, m: int = 8) -> float:
    """使用 m 个分点的复化 Simpson 公式计算定积分（调用 C 实现）"""
    _f = _CFUNC(f)
    return _lib.composite_simpson(_f, a, b, m)


def romberg(f: Callable[[float], float], a: float, b: float, tol: float = 1e-8) -> float:
    """使用 Romberg 方法计算定积分（调用 C 实现）"""
    _f = _CFUNC(f)
    return _lib.romberg(_f, a, b, tol)


def gauss_legendre(f: Callable[[float], float], a: float, b: float, n: int = 4) -> float:
    """使用 Gauss-Legendre 求积公式计算定积分（调用 C 实现）"""
    _f = _CFUNC(f)
    return _lib.gauss_legendre(_f, a, b, n)

---
## 第3题.

分别用复化梯形公式和复化 Simpson 公式计算下列积分

(2) $\displaystyle \int_0^{\pi} \dfrac{1 - \cos x}{x} ~ \mathrm{d} x,$ $m = 8$
(可看成连续函数 $\displaystyle f(x) = \begin{cases} \dfrac{1 - \cos x}{x} ~ \mathrm{d} x, & x \neq 0; \\ 0, & x = 0 \end{cases}$ 的积分);

(4) $\displaystyle \int_0^2 \dfrac{e^{-x}}{1+x^2} ~ \mathrm{d} x,$ $m = 8$.

In [ ]:
p3_f2 = lambda x: (1 - np.cos(x)) / x if x != 0 else 0
p3_f4 = lambda x: np.exp(-x) / (1 + x**2)

In [ ]:
"""Check that composite_trapezoid and composite_simpson return the correct output for problem (2)"""
mpm.mp.dps = 50


def p3_f2_mp(x):
    return (1 - mpm.cos(x)) / x if x != 0 else mpm.mpf("0")


p3_true_val_2 = mpm.quad(p3_f2_mp, [0, mpm.pi])
p3_true_val_2 = float(p3_true_val_2)

p3_approx_2 = composite_trapezoid(p3_f2, 0, np.pi, 8)
assert np.allclose(p3_approx_2, p3_true_val_2, atol=1e-2)

p3_approx_2 = composite_simpson(p3_f2, 0, np.pi, 8)
assert np.allclose(p3_approx_2, p3_true_val_2, atol=1e-4)

In [ ]:
"""Check that composite_trapezoid and composite_simpson return the correct output for problem (4)"""
mpm.mp.dps = 50


def p3_f4_mp(x):
    return mpm.e ** (-x) / (1 + x**2)


p3_true_val_4 = mpm.quad(p3_f4_mp, [0, 2])
p3_true_val_4 = float(p3_true_val_4)

p3_approx_4 = composite_trapezoid(p3_f4, 0, 2, 8)
assert np.allclose(p3_approx_4, p3_true_val_4, atol=1e-2)

p3_approx_4 = composite_simpson(p3_f4, 0, 2, 8)
assert np.allclose(p3_approx_4, p3_true_val_4, atol=1e-4)

In [ ]:
"""Hidden tests: check composite_trapezoid and composite_simpson on additional functions"""
### BEGIN HIDDEN TESTS

# Hidden function A: f(x) = exp(x) on [0, 1]  (exact = e-1)


def _h3_fa(x):
    return np.exp(x)


_h3_fa_true = np.e - 1

_h3_approx = composite_trapezoid(_h3_fa, 0, 1, 8)
assert np.allclose(_h3_approx, _h3_fa_true, atol=1e-2), f"composite_trapezoid on exp(x): {_h3_approx} vs {_h3_fa_true}"

_h3_approx = composite_simpson(_h3_fa, 0, 1, 8)
assert np.allclose(_h3_approx, _h3_fa_true, atol=1e-5), f"composite_simpson on exp(x): {_h3_approx} vs {_h3_fa_true}"

# Hidden function B: f(x) = sin(x) on [0, pi]  (exact = 2)


def _h3_fb(x):
    return np.sin(x)


_h3_approx = composite_trapezoid(_h3_fb, 0, np.pi, 8)
assert np.allclose(_h3_approx, 2.0, atol=5e-2), f"composite_trapezoid on sin(x): {_h3_approx} vs 2.0"

_h3_approx = composite_simpson(_h3_fb, 0, np.pi, 8)
assert np.allclose(_h3_approx, 2.0, atol=1e-4), f"composite_simpson on sin(x): {_h3_approx} vs 2.0"
### END HIDDEN TESTS

---

## 第4题.

用 Romberg 方法计算 $\displaystyle \int_1^2 \dfrac{\mathrm{d} x}{x},$ 精确到小数点后第8位

In [ ]:
p4_f = lambda x: 1 / x

In [ ]:
"""Check that romberg returns the correct answer"""
mpm.mp.dps = 50

p4_true_val = mpm.quad(p4_f, [1, 2])
p4_true_val = float(p4_true_val)

p4_approx = romberg(p4_f, 1, 2, 1e-8)
assert np.allclose(p4_approx, p4_true_val, atol=1e-8)

In [ ]:
"""Hidden tests: check romberg on additional functions"""
### BEGIN HIDDEN TESTS

# Hidden function A: f(x) = sin(x) on [0, pi]  (exact = 2)


def _h4_fa(x):
    return np.sin(x)


_h4_approx = romberg(_h4_fa, 0, np.pi, 1e-8)
assert np.allclose(_h4_approx, 2.0, atol=1e-8), f"romberg on sin(x): {_h4_approx} vs 2.0"

# Hidden function B: f(x) = exp(x) on [0, 1]  (exact = e-1)


def _h4_fb(x):
    return np.exp(x)


_h4_approx = romberg(_h4_fb, 0, 1, 1e-8)
assert np.allclose(_h4_approx, np.e - 1, atol=1e-8), f"romberg on exp(x): {_h4_approx} vs {np.e - 1}"
### END HIDDEN TESTS

---
## 第5题.

用一般的积分区间上的 Gauß-Legendre 公式 (取 $n = 4$) 计算积分 $\displaystyle I(N) = \int_0^N e^{-x^2} ~ \mathrm{d} x$:

(1) $N = 1$; &nbsp;&nbsp;&nbsp;&nbsp; (2) $N = 3$; &nbsp;&nbsp;&nbsp;&nbsp; (3) $N = 10$.

并与
$$\lim_{N\to\infty} \int_0^N e^{-x^2} ~ \mathrm{d} x = \dfrac{\sqrt{\pi}}{2}$$
的结果相比较

In [ ]:
p5_f = lambda x: np.exp(-(x**2))

In [ ]:
"""Check that gauss_legendre returns the correct answer for N=1,3;
   for N=10 (n=4), print the result to illustrate approximation error."""
mpm.mp.dps = 50


def p5_f_mp(x):
    return mpm.e ** (-(x**2))


for N in [1, 3]:
    p5_true_val = float(mpm.quad(p5_f_mp, [0, N]))
    p5_approx = gauss_legendre(p5_f, 0, N, 4)
    assert np.allclose(p5_approx, p5_true_val, atol=1e-2), f"{N=}: got {p5_approx=}, expected {p5_true_val=}"

# N=10: n=4 Gauss-Legendre is inaccurate; just display for comparison
p5_true_10 = float(mpm.quad(p5_f_mp, [0, 10]))
p5_approx_10 = gauss_legendre(p5_f, 0, 10, 4)
print(f"N=10: GL(n=4) = {p5_approx_10:.6f}, true = {p5_true_10:.6f}, error = {abs(p5_approx_10 - p5_true_10):.6f}")
print(f"√π/2 = {float(mpm.sqrt(mpm.pi)/2):.6f}")